# Reproducing the paper figures from the released traces

This notebook reproduces every paper figure derivable from the public
**GitHub Copilot for Visual Studio** agentic trace release, using the
paper's styling/plotting code (`paper_figures.py` + `paper_style.py`).

**No sampling is applied in the analysis** - every downloaded session/turn/call is
used (the release itself is already a uniformly sampled population of June 1-7,
2026).

**Prerequisites:** Download the trace shards from the
[GitHub release](https://github.com/Azure/AzurePublicDataset/releases/tag/ghcp-coding-agent-2026)
into the `downloaded_data/` folder, then `pip install -r requirements.txt`.
Run this notebook from this directory.

In [ ]:
import os
import glob
import pandas as pd
from IPython.display import Image, display, Markdown

import trace_loader
import trace_metrics
import paper_style
import paper_figures as F
import make_figures as M


## 1. Verify the local data

The trace shards should already be in `downloaded_data/` (downloaded from the
[GitHub release](https://github.com/Azure/AzurePublicDataset/releases/tag/ghcp-coding-agent-2026)).
The following cell checks that the shards are present.

In [ ]:
paths = trace_loader.shard_paths()
print(f"Found {len(paths)} shard file(s) in {trace_loader.DEFAULT_CACHE}")
assert len(paths) > 0, (
    "No shards found! Download the trace data from the GitHub release "
    "into downloaded_data/ first. See README.md for instructions."
)

## 2. Load the traces and describe the dataset scale

Flatten every session into tidy per-call / per-batch / per-turn tables, then
report the overall scale of the dataset before plotting anything.

In [ ]:
frames = trace_loader.load_dataframes()
summary = trace_metrics.scale_summary(frames)

labels = {
    "date_range": "Date range (UTC)",
    "n_days": "Days",
    "sessions": "Agentic sessions",
    "turns": "User turns",
    "llm_calls": "LLM calls",
    "tool_batches": "Tool batches",
    "tool_calls": "Individual tool calls",
    "distinct_models": "Distinct (user-facing) models",
    "prompt_tokens": "Prompt tokens (sum)",
    "completion_tokens": "Completion tokens (sum)",
    "cached_tokens": "Cached prompt tokens (sum)",
    "avg_turns_per_session": "Avg turns / session",
    "avg_llm_calls_per_turn": "Avg LLM calls / turn",
    "avg_tool_calls_per_turn": "Avg tool calls / turn",
}

def _fmt(v):
    return f"{v:,.0f}" if isinstance(v, float) and v == int(v) else (
        f"{v:,}" if isinstance(v, int) else str(v))

overview = pd.DataFrame(
    [(labels[k], _fmt(summary[k])) for k in labels],
    columns=["Metric", "Value"],
)
display(Markdown("### Dataset scale overview"))
display(overview.style.hide(axis="index"))

## 3. Aggregate the traces into the paper's intermediate tables

Re-derives every reproducible `data/*.csv` from the loaded traces, over the full
downloaded population.

In [ ]:
tables = trace_metrics.build_all(frames)
trace_metrics.write_all(tables, out_dir=M.DATA)
print(f"\n{len(tables)} tables written to {M.DATA}")

## 4. Render the figures

We reuse the paper's plotting code unchanged. `paper_style.save_fig` normally
writes a PDF; here we wrap it to *also* drop a PNG next to each PDF so the
figures can be shown inline below.

In [ ]:
PNG_DIR = os.path.join(M.OUT, "png")
os.makedirs(PNG_DIR, exist_ok=True)

_orig_save_fig = paper_style.save_fig

def _save_fig_with_png(fig, name, output_dir=None, pad_inches=0.0):
    path = _orig_save_fig(fig, name, output_dir=output_dir, pad_inches=pad_inches)
    try:
        fig.savefig(os.path.join(PNG_DIR, f"{name}.png"), format="png",
                    dpi=200, bbox_inches="tight", pad_inches=max(pad_inches, 0.02))
    except Exception as exc:  # fig may already be closed by _orig_save_fig
        print(f"  (png skipped for {name}: {exc})")
    return path

paper_style.save_fig = _save_fig_with_png

print("Group A (matplotlib defaults) ...")
M.group_a(M.OUT)
print("Group B (paper_style.setup_style) ...")
M.group_b(M.OUT)

paper_style.save_fig = _orig_save_fig
pdfs = sorted(glob.glob(os.path.join(M.OUT, "*.pdf")))
print(f"\nDone. {len(pdfs)} PDFs in {M.OUT} (PNG previews in {PNG_DIR}).")

## 5. Show the reproduced figures inline

In [ ]:
for png in sorted(glob.glob(os.path.join(PNG_DIR, "*.png"))):
    display(Markdown(f"**{os.path.splitext(os.path.basename(png))[0]}**"))
    display(Image(filename=png))

## Notes

* Figures reproduce the paper's **methodology and shape**, not its exact numbers:
  the release is a 25% sample of June 1-7, whereas some paper figures used a
  different day/window or the full population.
* `*_vs` figures are the Visual Studio series of the paper's VS-vs-VS-Code
  comparison plots.
* The traces are downloaded from the
  [GitHub release](https://github.com/Azure/AzurePublicDataset/releases/tag/ghcp-coding-agent-2026)
  and stored locally in `downloaded_data/`.